# 02_mlflow_model_registry

In [9]:
import sys
from pathlib import Path
import mlflow
import numpy as np

sys.executable

# Repo root (notebook is in /notebooks)
PROJECT_ROOT = Path.cwd().parent

# Dataset path
csv_path = PROJECT_ROOT / "data" / "raw" / "adult-census.csv"

# Everything under repo/mlflow/
MLFLOW_DIR = PROJECT_ROOT / "mlflow"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_DB = (MLFLOW_DIR / "mlflow.db").resolve()
ARTIFACT_ROOT = (MLFLOW_DIR / "artifacts").resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.as_posix()}")

EXPERIMENT_NAME = "adult_census_tracking"
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if exp is None:
    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=ARTIFACT_ROOT.as_uri(),
    )
mlflow.set_experiment(EXPERIMENT_NAME)

# Skore outputs (local, then logged to MLflow as artifacts)
SKORE_DIR = PROJECT_ROOT / "reports" / "skore"
SKORE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT  =", PROJECT_ROOT)
print("csv_path     =", csv_path)
print("exists?      =", csv_path.exists())
print("MLFLOW_DIR    =", MLFLOW_DIR)
print("TRACKING_URI  =", mlflow.get_tracking_uri())
print("MLFLOW_DB     =", MLFLOW_DB)
print("ARTIFACT_ROOT =", ARTIFACT_ROOT)
print("SKORE_DIR     =", SKORE_DIR)

PROJECT_ROOT  = H:\Documents\2. Perso\github\mlflow
csv_path     = H:\Documents\2. Perso\github\mlflow\data\raw\adult-census.csv
exists?      = True
MLFLOW_DIR    = H:\Documents\2. Perso\github\mlflow\mlflow
TRACKING_URI  = sqlite:///H:/Documents/2. Perso/github/mlflow/mlflow/mlflow.db
MLFLOW_DB     = H:\Documents\2. Perso\github\mlflow\mlflow\mlflow.db
ARTIFACT_ROOT = H:\Documents\2. Perso\github\mlflow\mlflow\artifacts
SKORE_DIR     = H:\Documents\2. Perso\github\mlflow\reports\skore


## Section 1 — Setup MLflow Registry

### Tracking ≠ Registry
- **Tracking** logs *runs* (params, metrics, artifacts): “what did I try and what happened?”
- **Registry** manages the *model lifecycle* (model name, versions, stages): “which model should be used in Staging/Production?”

### Why use a Registry even locally?
Because it gives you a **stable reference** (`models:/.../Production`) instead of hardcoding a `run_id`, and lets you practice **versioning, promotions, and rollbacks** in a production-like workflow, even with a local SQLite setup.

In [2]:
# Section 1 — Setup MLflow Registry (on top of your existing Tracking setup)

from mlflow.tracking import MlflowClient

REGISTRY_EXPERIMENT = "adult_census_registry"
mlflow.set_experiment(REGISTRY_EXPERIMENT)

client = MlflowClient()
MODEL_NAME = "adult_census_classifier"

print("REGISTRY_EXPERIMENT =", REGISTRY_EXPERIMENT)
print("TRACKING_URI        =", mlflow.get_tracking_uri())
print("MODEL_NAME          =", MODEL_NAME)

REGISTRY_EXPERIMENT = adult_census_registry
TRACKING_URI        = sqlite:///H:/Documents/2. Perso/github/mlflow/mlflow/mlflow.db
MODEL_NAME          = adult_census_classifier


## Section 2 — List versions (state of the world)

In [3]:
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
print("Found versions:", len(versions))

if not versions:
    raise RuntimeError(f"No versions found for model '{MODEL_NAME}'")

for v in versions:
    print(f"v{v.version} | stage={v.current_stage} | run_id={v.run_id}")

latest_version = str(max(int(v.version) for v in versions))
print("Latest version:", latest_version)

2026/01/16 10:15:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/16 10:15:00 INFO mlflow.store.db.utils: Updating database tables
2026/01/16 10:15:00 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/16 10:15:00 INFO alembic.runtime.migration: Will assume non-transactional DDL.


Found versions: 1
v1 | stage=Production | run_id=6adf9bfc23544540b515f095090d1f32
Latest version: 1


## Section 3 — Promote latest → Staging (logged)

In [4]:
with mlflow.start_run(run_name=f"promote_{MODEL_NAME}_v{latest_version}_to_staging"):
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=latest_version,
        stage="Staging",
    )
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("model_version", latest_version)
    mlflow.set_tag("action", "promote_to_staging")

print("Promoted to Staging:", latest_version)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_2784\3941252493.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Promoted to Staging: 1


## Section 4 — Promote latest → Production + load from Production (logged)

In [5]:
import mlflow.pyfunc

with mlflow.start_run(run_name=f"promote_{MODEL_NAME}_v{latest_version}_to_production_and_load"):
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=latest_version,
        stage="Production",
        archive_existing_versions=True,
    )
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("model_version", latest_version)
    mlflow.set_tag("action", "promote_to_production")

    model_prod = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/Production")
    mlflow.set_tag("loaded_uri", f"models:/{MODEL_NAME}/Production")

print(f"✓ {MODEL_NAME} v{latest_version} promoted to Production")
print("✓ Loaded model from Production")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_2784\1752371037.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


✓ adult_census_classifier v1 promoted to Production
✓ Loaded model from Production


## Section 5 — Sanity check inference (self-contained)

In [10]:
# Section 5 — Sanity check inference
# ------------------------------------------------------------------------------
# Goal:
# - Reproduce the exact same dataset preparation as in notebook 01_baseline_sklearn_pipeline.ipynb
# - Load the model from the MLflow Registry "Production" stage
# - Run a quick inference sanity check on a few rows (and optionally on the full test set)

import pandas as pd
import mlflow.pyfunc
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

MODEL_NAME = "adult_census_classifier"

# 1) Load the model from the Registry (Production stage)
model_prod = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/Production")
print(f"Loaded model from: models:/{MODEL_NAME}/Production")

# 2) Load dataset and apply the exact same preprocessing as Notebook #1
adult_census = pd.read_csv(csv_path)

# Notebook #1 removes this column (keep it identical)
adult_census = adult_census.drop(columns="education.num")

# Define target + features exactly like Notebook #1
target_name = "income"
target = adult_census[target_name]
data = adult_census.drop(columns=[target_name])

# Encode target with the exact mapping used in Notebook #1
target_map = {"<=50K": 0, ">50K": 1}
target_enc = target.map(target_map).astype("int64")

# 3) Use the same train/test split configuration (random_state + stratify)
data_train, data_test, target_train, target_test = train_test_split(
    data,
    target_enc,
    test_size=0.2,
    random_state=42,
    stratify=target_enc,
)

# 4) Sanity check: predict on a small sample
preds_sample = model_prod.predict(data_test.head(10))
print("Sample preds :", np.asarray(preds_sample))
print("Sample y_true:", target_test.head(10).to_numpy())

# 5) Quick overall accuracy on the test split
preds_all = model_prod.predict(data_test)
acc = accuracy_score(target_test, preds_all)
print(f"Sanity test accuracy: {acc:.4f}")

Loaded model from: models:/adult_census_classifier/Production
Sample preds : [0 0 1 0 0 1 0 0 0 0]
Sample y_true: [0 0 1 1 0 0 0 0 0 0]
Sanity test accuracy: 0.8544


## Section 6 — Create a v2 (new model version) and log/register it

In [11]:
# Section 6 — Train + Register a v2 (LogReg) as a new Model Version in the Registry
# ------------------------------------------------------------------------------
# Goal:
# - Train a new sklearn pipeline (v2)
# - Log metrics + params
# - Register it under the SAME Registered Model name (MODEL_NAME)
#   => this creates a NEW model version (v2) in the MLflow Registry

import numpy as np
import mlflow
import mlflow.sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# We reuse the exact same split variables from Notebook #1 / Section 5:
# data_train, data_test, target_train, target_test

# 1) Detect numerical vs categorical columns on the training set
num_cols = data_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in data_train.columns if c not in num_cols]

# 2) Build preprocessing pipelines
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
)

# 3) Define v2 model (Logistic Regression)
v2_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=500)),
])

# 4) Train + log + register
with mlflow.start_run(run_name="train_register_v2_logreg") as run:
    # Fit
    v2_pipeline.fit(data_train, target_train)

    # Predict
    y_pred = v2_pipeline.predict(data_test)

    # Metrics
    acc = accuracy_score(target_test, y_pred)
    mlflow.log_metric("test_accuracy", acc)

    # AUC (only for binary classification + proba available)
    if hasattr(v2_pipeline, "predict_proba") and len(np.unique(target_test)) == 2:
        y_proba = v2_pipeline.predict_proba(data_test)[:, 1]
        auc = roc_auc_score(target_test, y_proba)
        mlflow.log_metric("test_roc_auc", auc)

    # Params (useful for UI comparison)
    mlflow.log_param("model_family", "logreg")
    mlflow.log_param("max_iter", 500)
    mlflow.log_param("n_num_cols", len(num_cols))
    mlflow.log_param("n_cat_cols", len(cat_cols))

    # Register as a NEW version under the same Registered Model
    mlflow.sklearn.log_model(
        sk_model=v2_pipeline,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
    )

print("✓ v2 logged + registered")
print("Run ID:", run.info.run_id)
print("Model:", MODEL_NAME)
print("Check MLflow UI > Models to see the new version")

2026/01/16 10:29:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'adult_census_classifier' already exists. Creating a new version of this model...
Created version '2' of model 'adult_census_classifier'.


✓ v2 logged + registered
Run ID: 5da6635ebf7a44d68c54ad54ca9d0bb4
Model: adult_census_classifier
Check MLflow UI > Models to see the new version


## Section 7 — Find v2 version number, promote it to Production, then rollback to v1

In [12]:
# Section 7 — Promote v2 to Production, then rollback to v1 (Registry lifecycle)
# ------------------------------------------------------------------------------
# Goal:
# - Fetch all model versions from the Registry
# - Identify v1 (oldest) and v2 (latest)
# - Promote v2 to Production (and archive existing Production)
# - Rollback to v1 (promote v1 back to Production)
#
# This section does NOT train anything.
# It only manipulates lifecycle stages in the MLflow Model Registry.

from mlflow.tracking import MlflowClient

client = MlflowClient()

# Fetch all model versions for the registered model
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
all_versions = sorted({int(v.version) for v in versions})

print("All versions found:", all_versions)

# Safety check: we need at least 2 versions to demonstrate promotion + rollback
if len(all_versions) < 2:
    raise RuntimeError(
        f"Need at least 2 model versions for '{MODEL_NAME}' to run promotion + rollback."
    )

# v1 = oldest, v2 = latest
v1 = str(min(all_versions))
v2 = str(max(all_versions))

print("v1 (oldest) =", v1)
print("v2 (latest) =", v2)

All versions found: [1, 2]
v1 (oldest) = 1
v2 (latest) = 2


In [13]:
# Promote v2 to Production (archive any currently-Production versions)
# ------------------------------------------------------------------------------
# We log this operation as a dedicated run in the "adult_census_registry" experiment,
# to keep lifecycle actions separated from training runs.

import mlflow

mlflow.set_experiment("adult_census_registry")

with mlflow.start_run(run_name=f"promote_{MODEL_NAME}_v{v2}_to_production"):
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=v2,
        stage="Production",
        archive_existing_versions=True,
    )

    # Minimal metadata for traceability
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("model_version", v2)
    mlflow.set_tag("action", "promote_to_production")

print(f"✓ {MODEL_NAME} v{v2} is now Production (previous Production version archived)")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_2784\860144185.py:11: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


✓ adult_census_classifier v2 is now Production (previous Production version archived)


In [14]:
# Rollback to v1 (promote v1 back to Production)
# ------------------------------------------------------------------------------
# This simulates a real incident response: v2 had issues in Production,
# so we revert to the last known good version (v1).

with mlflow.start_run(run_name=f"rollback_{MODEL_NAME}_to_v{v1}"):
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=v1,
        stage="Production",
        archive_existing_versions=True,
    )

    # Minimal metadata for traceability
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("model_version", v1)
    mlflow.set_tag("action", "rollback_to_v1")

print(f"✓ Rollback complete: {MODEL_NAME} v{v1} is back to Production")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_2784\2389825216.py:7: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


✓ Rollback complete: adult_census_classifier v1 is back to Production


In [16]:
# Smoke test after rollback: load the Production model from the Registry
# ------------------------------------------------------------------------------
# In a real service, inference code should only ever depend on:
#   models:/<model_name>/Production
# not on a run_id.

import mlflow.pyfunc

model_prod = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/Production")
print(f"✓ Loaded model from: models:/{MODEL_NAME}/Production")

✓ Loaded model from: models:/adult_census_classifier/Production
